
# Bike-Lane Hazard Detection — Custom CNN Baseline

This notebook is a clean starting point for the project's **custom CNN** model.

It is designed to:

1. Detect multiple hazards in one image with bounding boxes.
2. Assign each detected hazard a severity from **1 to 5**.
3. Return an image-level danger score from **0 to 5** by averaging the accepted detections.
4. Treat an image with no accepted hazard detections as danger score **0**.
5. Train from CVAT annotations exported in **COCO 1.0** format.
6. Produce object-detection metrics, severity-aware metrics, example predictions, and a hazard heatmap.

> Important labeling rule: **“Good / safe” is not a bounding-box class.**  
> A safe image has no hazard boxes and receives an image danger score of 0.

The five box classes are:

| COCO category ID | Hazard class | Severity |
|---:|---|---:|
| 1 | Small obstructions | 1 |
| 2 | Poor maintenance / faded paint | 2 |
| 3 | Large objects / debris | 3 |
| 4 | Uneven ground / cracks / utility covers | 4 |
| 5 | Potholes | 5 |

This is an object detector rather than a whole-image classifier because one image may contain several hazards.



## Recommended CVAT export structure

In CVAT, annotate hazards with **rectangles** and export each split as **COCO 1.0**.

Prepare the data like this:

```text
bike_lane_data/
├── train/
│   ├── images/
│   └── annotations.json
├── val/
│   ├── images/
│   └── annotations.json
└── test/
    ├── images/
    └── annotations.json
```

### Split before exporting

Keep every frame from the same video and every image from the same physical location in only one split. A reasonable starting split is:

- 70% of locations/videos for training
- 15% for validation
- 15% drone-only locations/videos for testing

Do **not** randomly split consecutive video frames. That would make nearly identical frames appear in training and validation/testing and inflate the results.


In [ ]:

# Run in Google Colab or a fresh environment.
# Restart the runtime after installation if Colab asks you to.

%pip install -q torch torchvision torchmetrics pycocotools matplotlib pandas pillow


In [ ]:

from __future__ import annotations

import json
import math
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:

# ---------------------------
# Project configuration
# ---------------------------

DATA_ROOT = Path("/content/bike_lane_data")  # Change this path.

TRAIN_IMAGES = DATA_ROOT / "train" / "images"
TRAIN_ANNOTATIONS = DATA_ROOT / "train" / "annotations.json"

VAL_IMAGES = DATA_ROOT / "val" / "images"
VAL_ANNOTATIONS = DATA_ROOT / "val" / "annotations.json"

TEST_IMAGES = DATA_ROOT / "test" / "images"
TEST_ANNOTATIONS = DATA_ROOT / "test" / "annotations.json"

OUTPUT_DIR = Path("/content/bike_lane_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    0: "background",
    1: "small_obstruction",
    2: "poor_maintenance",
    3: "large_object",
    4: "uneven_ground",
    5: "pothole",
}

# Faster R-CNN includes background internally, so there are 6 model classes:
# background + five hazard classes.
NUM_MODEL_CLASSES = 6

SEVERITY = {
    1: 1.0,
    2: 2.0,
    3: 3.0,
    4: 4.0,
    5: 5.0,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

BATCH_SIZE = 2
NUM_WORKERS = 2
LEARNING_RATE = 0.005
WEIGHT_DECAY = 0.0005
NUM_EPOCHS = 20

# Only detections at or above this confidence contribute to the danger score.
SCORE_THRESHOLD = 0.50

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)



## Dataset loader

The loader below reads a standard COCO JSON file and returns the format required by Torchvision detection models:

```python
image, {
    "boxes": FloatTensor[N, 4],
    "labels": Int64Tensor[N],
    ...
}
```

Safe images are allowed to contain zero boxes.


In [ ]:

class CocoBikeLaneDataset(Dataset):
    def __init__(
        self,
        images_dir: Path | str,
        annotation_file: Path | str,
        training: bool = False,
    ) -> None:
        self.images_dir = Path(images_dir)
        self.annotation_file = Path(annotation_file)
        self.training = training

        if not self.annotation_file.exists():
            raise FileNotFoundError(f"Annotation file not found: {self.annotation_file}")
        if not self.images_dir.exists():
            raise FileNotFoundError(f"Image directory not found: {self.images_dir}")

        with self.annotation_file.open("r", encoding="utf-8") as f:
            coco = json.load(f)

        self.images = sorted(coco["images"], key=lambda x: x["id"])
        self.annotations_by_image: Dict[int, List[dict]] = defaultdict(list)
        for ann in coco.get("annotations", []):
            self.annotations_by_image[int(ann["image_id"])].append(ann)

        # CVAT category IDs should already be 1..5.
        category_ids = sorted(int(cat["id"]) for cat in coco.get("categories", []))
        expected = [1, 2, 3, 4, 5]
        if category_ids and category_ids != expected:
            raise ValueError(
                f"Expected COCO category IDs {expected}, but found {category_ids}. "
                "Fix the category IDs in CVAT/export before training."
            )

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int):
        info = self.images[index]
        image_id = int(info["id"])
        image_path = self.images_dir / info["file_name"]

        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        boxes: List[List[float]] = []
        labels: List[int] = []
        areas: List[float] = []
        iscrowd: List[int] = []

        for ann in self.annotations_by_image.get(image_id, []):
            x, y, w, h = map(float, ann["bbox"])

            # Clamp boxes to image bounds and reject invalid boxes.
            x1 = max(0.0, min(x, width - 1))
            y1 = max(0.0, min(y, height - 1))
            x2 = max(0.0, min(x + w, width))
            y2 = max(0.0, min(y + h, height))

            if x2 <= x1 or y2 <= y1:
                continue

            label = int(ann["category_id"])
            if label not in SEVERITY:
                raise ValueError(f"Unexpected category ID {label} in image {image_id}")

            boxes.append([x1, y1, x2, y2])
            labels.append(label)
            areas.append((x2 - x1) * (y2 - y1))
            iscrowd.append(int(ann.get("iscrowd", 0)))

        boxes_tensor = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels_tensor = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes_tensor,
            "labels": labels_tensor,
            "image_id": torch.tensor(image_id, dtype=torch.int64),
            "area": torch.tensor(areas, dtype=torch.float32),
            "iscrowd": torch.tensor(iscrowd, dtype=torch.int64),
        }

        # Safe geometric augmentation: horizontal flip only.
        # Add more transformations only after checking that they preserve realistic road views.
        if self.training and random.random() < 0.5:
            image = F.hflip(image)
            if len(boxes_tensor) > 0:
                flipped = boxes_tensor.clone()
                flipped[:, [0, 2]] = width - boxes_tensor[:, [2, 0]]
                target["boxes"] = flipped

        image_tensor = F.pil_to_tensor(image).float() / 255.0
        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


In [ ]:

def validate_coco_dataset(dataset: CocoBikeLaneDataset) -> pd.DataFrame:
    rows = []
    class_counts = defaultdict(int)
    safe_images = 0

    for idx in range(len(dataset)):
        image, target = dataset[idx]
        labels = target["labels"].tolist()

        if len(labels) == 0:
            safe_images += 1

        for label in labels:
            class_counts[int(label)] += 1

        rows.append({
            "index": idx,
            "image_id": int(target["image_id"]),
            "width": int(image.shape[2]),
            "height": int(image.shape[1]),
            "box_count": int(len(labels)),
            "ground_truth_danger": (
                float(np.mean([SEVERITY[int(x)] for x in labels]))
                if labels else 0.0
            ),
        })

    summary = pd.DataFrame({
        "class_id": list(range(0, 6)),
        "class_name": ["safe_image"] + [CLASS_NAMES[i] for i in range(1, 6)],
        "count": [safe_images] + [class_counts[i] for i in range(1, 6)],
    })

    print(f"Images: {len(dataset)}")
    print(f"Safe images with zero boxes: {safe_images}")
    display(summary)
    return pd.DataFrame(rows)


# Run after the files have been placed at the configured paths:
#
# train_dataset = CocoBikeLaneDataset(TRAIN_IMAGES, TRAIN_ANNOTATIONS, training=True)
# val_dataset = CocoBikeLaneDataset(VAL_IMAGES, VAL_ANNOTATIONS, training=False)
# test_dataset = CocoBikeLaneDataset(TEST_IMAGES, TEST_ANNOTATIONS, training=False)
#
# train_index = validate_coco_dataset(train_dataset)



## Custom CNN backbone + Faster R-CNN

A plain CNN classifier cannot return several bounding boxes. This baseline therefore uses:

- a **custom CNN backbone** to learn image features;
- a **region proposal network** to propose candidate hazard locations;
- classification and box-regression heads to label and refine each detection.

The later ResNet-18 and EfficientNet-B0 experiments can use the same data, training loop, scoring rule, and evaluation code while changing only the backbone.


In [ ]:

class CustomCNNBackbone(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        def block(in_channels: int, out_channels: int) -> nn.Sequential:
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2),
            )

        self.features = nn.Sequential(
            block(3, 32),
            block(32, 64),
            block(64, 128),
            block(128, 256),
        )

        # Required by Torchvision FasterRCNN.
        self.out_channels = 256

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


def build_custom_cnn_detector(num_classes: int = NUM_MODEL_CLASSES) -> FasterRCNN:
    backbone = CustomCNNBackbone()

    anchor_generator = AnchorGenerator(
        sizes=((16, 32, 64, 128, 256),),
        aspect_ratios=((0.5, 1.0, 2.0),),
    )

    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=["0"],
        output_size=7,
        sampling_ratio=2,
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
        min_size=512,
        max_size=1024,
        box_score_thresh=0.05,
    )
    return model


model = build_custom_cnn_detector().to(DEVICE)
print(type(model).__name__)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:

# Optional smoke test: verifies that the model accepts an image.
# It does not train the model or provide meaningful predictions.

model.eval()
with torch.no_grad():
    dummy_image = torch.rand(3, 512, 512, device=DEVICE)
    dummy_output = model([dummy_image])[0]

print({key: tuple(value.shape) for key, value in dummy_output.items()})


## Data loaders

In [ ]:

def make_loaders():
    train_dataset = CocoBikeLaneDataset(
        TRAIN_IMAGES, TRAIN_ANNOTATIONS, training=True
    )
    val_dataset = CocoBikeLaneDataset(
        VAL_IMAGES, VAL_ANNOTATIONS, training=False
    )
    test_dataset = CocoBikeLaneDataset(
        TEST_IMAGES, TEST_ANNOTATIONS, training=False
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=torch.cuda.is_available(),
    )
    return train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader


# Uncomment when the exported CVAT data is ready:
#
# (
#     train_dataset,
#     val_dataset,
#     test_dataset,
#     train_loader,
#     val_loader,
#     test_loader,
# ) = make_loaders()



## Training loop

The model returns four losses during training:

- classification loss;
- bounding-box regression loss;
- region-proposal objectness loss;
- region-proposal box regression loss.

The validation loss is calculated by temporarily placing the model in training mode while disabling gradients. Batch-normalization layers remain frozen during that calculation.


In [ ]:

def move_targets_to_device(targets, device):
    return [
        {key: value.to(device) if torch.is_tensor(value) else value
         for key, value in target.items()}
        for target in targets
    ]


def train_one_epoch(model, loader, optimizer, device) -> float:
    model.train()
    running_loss = 0.0

    for images, targets in loader:
        images = [image.to(device) for image in images]
        targets = move_targets_to_device(targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss detected: {loss.item()} | {loss_dict}")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        running_loss += float(loss.item())

    return running_loss / max(len(loader), 1)


@torch.no_grad()
def calculate_validation_loss(model, loader, device) -> float:
    model.train()

    # Freeze BatchNorm running statistics while still allowing detection losses.
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    for images, targets in loader:
        images = [image.to(device) for image in images]
        targets = move_targets_to_device(targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        running_loss += float(loss.item())

    model.eval()
    return running_loss / max(len(loader), 1)


def fit_model(model, train_loader, val_loader, epochs=NUM_EPOCHS):
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
        momentum=0.9,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=5, gamma=0.2
    )

    history = []
    best_val_loss = float("inf")
    best_path = OUTPUT_DIR / "best_custom_cnn_detector.pth"

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
        val_loss = calculate_validation_loss(model, val_loader, DEVICE)
        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train loss={train_loss:.4f} | "
            f"val loss={val_loss:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "class_names": CLASS_NAMES,
                    "severity": SEVERITY,
                    "epoch": epoch,
                    "val_loss": val_loss,
                },
                best_path,
            )

    history_df = pd.DataFrame(history)
    history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
    print("Best checkpoint:", best_path)
    return history_df


# Train after the data loaders exist:
#
# history = fit_model(model, train_loader, val_loader)


In [ ]:

def plot_training_history(history: pd.DataFrame) -> None:
    plt.figure(figsize=(8, 5))
    plt.plot(history["epoch"], history["train_loss"], label="Training loss")
    plt.plot(history["epoch"], history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Custom CNN Detector Training")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


# plot_training_history(history)



## Danger-score calculation

For accepted detections with severities \(s_1, s_2, \dots, s_n\):

\[
\text{danger score} = \frac{s_1 + s_2 + \cdots + s_n}{n}
\]

If no detection passes the confidence threshold, the score is 0.

The function also returns a confidence-weighted score as an optional secondary result. The unweighted average is the project's main score.


In [ ]:

def calculate_danger_score(
    prediction: Dict[str, torch.Tensor],
    score_threshold: float = SCORE_THRESHOLD,
) -> Dict[str, object]:
    scores = prediction["scores"].detach().cpu()
    labels = prediction["labels"].detach().cpu()
    boxes = prediction["boxes"].detach().cpu()

    keep = scores >= score_threshold
    kept_scores = scores[keep]
    kept_labels = labels[keep]
    kept_boxes = boxes[keep]

    severities = torch.tensor(
        [SEVERITY[int(label)] for label in kept_labels.tolist()],
        dtype=torch.float32,
    )

    if len(severities) == 0:
        danger = 0.0
        confidence_weighted = 0.0
    else:
        danger = float(severities.mean().item())
        confidence_weighted = float(
            (severities * kept_scores).sum().item() / kept_scores.sum().item()
        )

    return {
        "danger_score": round(danger, 3),
        "confidence_weighted_score": round(confidence_weighted, 3),
        "hazard_count": int(len(kept_labels)),
        "labels": kept_labels,
        "scores": kept_scores,
        "boxes": kept_boxes,
    }


def ground_truth_danger_score(target: Dict[str, torch.Tensor]) -> float:
    labels = target["labels"].detach().cpu().tolist()
    if not labels:
        return 0.0
    return float(np.mean([SEVERITY[int(label)] for label in labels]))



## Evaluation

Use several measurements because “accuracy” alone is not sufficient for object detection:

- **mAP@0.50:0.95** and **mAP@0.50** for boxes and classes;
- per-image danger-score mean absolute error;
- severe-miss rate: how often a truly severe image is predicted as relatively safe;
- severity-weighted matched-class error;
- precision and recall from the mAP evaluator.

The severe-miss rule below counts a miss when the true danger score is at least 4 but the predicted score is below 2.


In [ ]:

def match_predictions_to_targets(
    prediction: Dict[str, torch.Tensor],
    target: Dict[str, torch.Tensor],
    score_threshold: float = SCORE_THRESHOLD,
    iou_threshold: float = 0.5,
) -> List[Tuple[int, int]]:
    pred_scores = prediction["scores"].detach().cpu()
    pred_boxes = prediction["boxes"].detach().cpu()[pred_scores >= score_threshold]
    pred_labels = prediction["labels"].detach().cpu()[pred_scores >= score_threshold]

    true_boxes = target["boxes"].detach().cpu()
    true_labels = target["labels"].detach().cpu()

    if len(pred_boxes) == 0 or len(true_boxes) == 0:
        return []

    ious = box_iou(pred_boxes, true_boxes)
    matches = []
    used_predictions = set()
    used_targets = set()

    # Greedy highest-IoU matching.
    flat_indices = torch.argsort(ious.flatten(), descending=True)
    for flat_index in flat_indices.tolist():
        pred_idx = flat_index // ious.shape[1]
        true_idx = flat_index % ious.shape[1]
        iou = float(ious[pred_idx, true_idx])

        if iou < iou_threshold:
            break
        if pred_idx in used_predictions or true_idx in used_targets:
            continue

        used_predictions.add(pred_idx)
        used_targets.add(true_idx)
        matches.append((int(pred_labels[pred_idx]), int(true_labels[true_idx])))

    return matches


@torch.no_grad()
def evaluate_model(model, loader, device=DEVICE) -> Tuple[dict, pd.DataFrame]:
    model.eval()
    map_metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")

    image_rows = []
    matched_class_errors = []

    for images, targets in loader:
        device_images = [image.to(device) for image in images]
        outputs = model(device_images)

        cpu_outputs = [
            {key: value.detach().cpu() for key, value in output.items()}
            for output in outputs
        ]
        cpu_targets = [
            {key: value.detach().cpu() if torch.is_tensor(value) else value
             for key, value in target.items()}
            for target in targets
        ]

        map_metric.update(cpu_outputs, cpu_targets)

        for output, target in zip(cpu_outputs, cpu_targets):
            score_info = calculate_danger_score(output)
            predicted_danger = float(score_info["danger_score"])
            true_danger = ground_truth_danger_score(target)

            image_rows.append({
                "image_id": int(target["image_id"]),
                "true_danger": true_danger,
                "predicted_danger": predicted_danger,
                "absolute_error": abs(predicted_danger - true_danger),
                "true_hazard_count": int(len(target["labels"])),
                "predicted_hazard_count": int(score_info["hazard_count"]),
                "severe_miss": int(true_danger >= 4.0 and predicted_danger < 2.0),
            })

            for predicted_label, true_label in match_predictions_to_targets(output, target):
                matched_class_errors.append(abs(predicted_label - true_label))

    detection_metrics = map_metric.compute()
    image_metrics = pd.DataFrame(image_rows)

    summary = {
        "map_50_95": float(detection_metrics["map"]),
        "map_50": float(detection_metrics["map_50"]),
        "mar_100": float(detection_metrics["mar_100"]),
        "danger_mae": float(image_metrics["absolute_error"].mean()),
        "severe_miss_rate": float(image_metrics["severe_miss"].mean()),
        "matched_severity_error": (
            float(np.mean(matched_class_errors))
            if matched_class_errors else float("nan")
        ),
    }
    return summary, image_metrics


# Example:
#
# checkpoint = torch.load(
#     OUTPUT_DIR / "best_custom_cnn_detector.pth",
#     map_location=DEVICE,
# )
# model.load_state_dict(checkpoint["model_state_dict"])
# test_summary, test_predictions = evaluate_model(model, test_loader)
# print(test_summary)
# display(test_predictions.head())
# test_predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)


## Visualize detections and the final danger score

In [ ]:

@torch.no_grad()
def predict_image(
    model,
    image_path: Path | str,
    device=DEVICE,
    score_threshold: float = SCORE_THRESHOLD,
):
    image = Image.open(image_path).convert("RGB")
    tensor = F.pil_to_tensor(image).float() / 255.0

    model.eval()
    prediction = model([tensor.to(device)])[0]
    result = calculate_danger_score(prediction, score_threshold)

    return image, prediction, result


def draw_prediction(
    image: Image.Image,
    result: Dict[str, object],
    line_width: int = 4,
) -> Image.Image:
    rendered = image.copy()
    draw = ImageDraw.Draw(rendered)

    boxes = result["boxes"].tolist()
    labels = result["labels"].tolist()
    scores = result["scores"].tolist()

    for box, label, confidence in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        draw.rectangle([x1, y1, x2, y2], width=line_width)
        text = f"{CLASS_NAMES[int(label)]} | severity {label} | {confidence:.2f}"
        draw.text((x1 + 4, max(0, y1 - 16)), text)

    title = (
        f"Danger score: {result['danger_score']:.2f}/5 | "
        f"Hazards: {result['hazard_count']}"
    )
    draw.text((10, 10), title)
    return rendered


# Example:
#
# image, prediction, result = predict_image(
#     model, TEST_IMAGES / "example.jpg"
# )
# rendered = draw_prediction(image, result)
# display(rendered)
# print(result)



## Hazard heatmap

This heatmap shows where the detector places danger. Each accepted bounding box adds its severity multiplied by its confidence to the covered area. It is easier to interpret for this project than a classification-only heatmap because it directly corresponds to detected hazards.


In [ ]:

def create_hazard_heatmap(
    image: Image.Image,
    result: Dict[str, object],
    alpha: float = 0.45,
) -> None:
    width, height = image.size
    heat = np.zeros((height, width), dtype=np.float32)

    boxes = result["boxes"].tolist()
    labels = result["labels"].tolist()
    scores = result["scores"].tolist()

    for box, label, confidence in zip(boxes, labels, scores):
        x1, y1, x2, y2 = map(int, box)
        x1, x2 = np.clip([x1, x2], 0, width)
        y1, y2 = np.clip([y1, y2], 0, height)

        if x2 > x1 and y2 > y1:
            heat[y1:y2, x1:x2] += SEVERITY[int(label)] * float(confidence)

    plt.figure(figsize=(12, 7))
    plt.imshow(image)
    plt.imshow(heat, alpha=alpha, cmap="hot")
    plt.colorbar(label="Predicted hazard intensity")
    plt.axis("off")
    plt.title(f"Hazard Heatmap — Danger {result['danger_score']:.2f}/5")
    plt.show()


# Example:
# create_hazard_heatmap(image, result)



## Confidence-threshold selection

The confidence threshold should not be chosen from the test set. Use the validation set to test several thresholds, then lock one threshold before final drone-only testing.

This avoids quietly choosing a threshold that makes the final test results look better.


In [ ]:

@torch.no_grad()
def compare_thresholds(
    model,
    loader,
    thresholds=(0.25, 0.35, 0.50, 0.65, 0.75),
    device=DEVICE,
) -> pd.DataFrame:
    model.eval()
    cached = []

    for images, targets in loader:
        outputs = model([image.to(device) for image in images])
        for output, target in zip(outputs, targets):
            cached.append((
                {k: v.detach().cpu() for k, v in output.items()},
                {k: v.detach().cpu() if torch.is_tensor(v) else v
                 for k, v in target.items()},
            ))

    rows = []
    for threshold in thresholds:
        errors = []
        severe_misses = []
        predicted_counts = []

        for output, target in cached:
            score_info = calculate_danger_score(output, threshold)
            pred = float(score_info["danger_score"])
            true = ground_truth_danger_score(target)

            errors.append(abs(pred - true))
            severe_misses.append(int(true >= 4.0 and pred < 2.0))
            predicted_counts.append(int(score_info["hazard_count"]))

        rows.append({
            "threshold": threshold,
            "danger_mae": float(np.mean(errors)),
            "severe_miss_rate": float(np.mean(severe_misses)),
            "mean_predicted_hazards": float(np.mean(predicted_counts)),
        })

    return pd.DataFrame(rows)


# threshold_results = compare_thresholds(model, val_loader)
# display(threshold_results)



## Save and reload the final model


In [ ]:

FINAL_MODEL_PATH = OUTPUT_DIR / "custom_cnn_bike_lane_detector.pth"

def save_final_model(model, path=FINAL_MODEL_PATH):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "class_names": CLASS_NAMES,
            "severity": SEVERITY,
            "score_threshold": SCORE_THRESHOLD,
        },
        path,
    )
    print("Saved:", path)


def load_final_model(path=FINAL_MODEL_PATH, device=DEVICE):
    checkpoint = torch.load(path, map_location=device)
    loaded_model = build_custom_cnn_detector().to(device)
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model.eval()
    return loaded_model, checkpoint


# save_final_model(model)
# loaded_model, metadata = load_final_model()



# Annotation rules to finalize before training

Use the same written rules for every annotator.

### Class 1 — Small obstructions
Examples: limited vegetation encroachment, small loose items, minor gravel patches.

### Class 2 — Poor maintenance
Examples: faded or missing lane paint, unclear bike-lane markings, badly maintained visual separation.

### Class 3 — Large objects / debris
Examples: trash bags, branches, cones, bins, parked objects, substantial debris blocking travel.

### Class 4 — Uneven ground
Examples: cracks, broken asphalt, raised or sunken utility covers, abrupt surface-level changes.

### Class 5 — Potholes
Examples: visible pavement cavities or holes that could catch a bicycle wheel.

### Required consistency decisions

1. Draw one box around each separate hazard.
2. Do not draw a “good” box.
3. When one physical hazard could fit two classes, use the more specific class.
4. Potholes always use class 5 rather than class 4.
5. A faded lane marking is class 2, not class 0.
6. A bike lane that is hard to identify because of poor paint may be class 2.
7. Do not label an image as class 2 merely because the camera angle is poor.
8. Decide whether one continuous crack is one box or several boxes, and apply that rule everywhere.
9. Review a shared sample set together before annotating the full dataset.
10. Track the source video and location for every frame so the splits remain independent.



# Next steps

1. Extract useful frames from the seven videos.
2. Remove near-duplicate frames rather than labeling every frame.
3. Assign each video/location to train, validation, or test before model training.
4. Annotate rectangles in CVAT using category IDs 1–5.
5. Include genuinely safe images with no boxes.
6. Export each split as COCO 1.0.
7. Update `DATA_ROOT`.
8. Run dataset validation.
9. Train the custom CNN detector.
10. Select the confidence threshold using validation data.
11. Evaluate once on the held-out drone-only test set.

For the later model comparison, keep the same split, annotation files, augmentations, number of epochs, threshold-selection procedure, and metrics for the custom CNN, ResNet-18, and EfficientNet-B0.
